#  UrbanNest Analytics — House Rent Prediction Pipeline

**Task 1: Model Training & Hyperparameter Optimization**

This notebook implements a complete end-to-end ML pipeline for predicting house rent prices.
It covers:

1. **Data Loading & Exploration** — Understanding the dataset
2. **Preprocessing Pipeline** — LabelEncoding all categorical features
3. **Experiment Tracking** — Using `trackio` to log all experiments
4. **Hyperparameter Optimization** — Grid Search, Random Search, Bayesian (Optuna)
5. **Convergence Analysis & Plots** — Visualizing the search strategies
6. **Final Model Training & Evaluation** — Best model on test set

---
**Dataset:** Indian House Rent dataset (Mumbai, Delhi, Pune)  
**Target:** `price` (monthly rent in INR)  
**Model:** RandomForestRegressor  
**Random State:** 42 (for full reproducibility)


---
## Section 1: Setup & Imports

Import all required libraries and configure global settings.

In [153]:
# ── Standard library ──────────────────────────────────────────────
import os
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Machine learning ──────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score,
    KFold
)
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint
import joblib
from joblib import parallel_backend

# ── Hyperparameter optimization ───────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress verbose logs

# ── Experiment tracking ───────────────────────────────────────────
import trackio

# ── Visualization ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Style settings ────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'figure.dpi': 140,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})

# ── Reproducibility ───────────────────────────────────────────────
RANDOM_STATE = 42
N_JOBS = -1
PARALLEL_BACKEND = 'threading'
np.random.seed(RANDOM_STATE)

# ── Directory setup ───────────────────────────────────────────────
os.makedirs('models', exist_ok=True)
os.makedirs('plots',  exist_ok=True)

print('✅ All imports successful!')
print(f'   NumPy     : {np.__version__}')
print(f'   Pandas    : {pd.__version__}')
print(f'   Optuna    : {optuna.__version__}')
print(f'   SKLearn   : {__import__("sklearn").__version__}')


✅ All imports successful!
   NumPy     : 2.4.2
   Pandas    : 2.3.3
   Optuna    : 4.4.0
   SKLearn   : 1.7.1


---
## 📊 Section 2: Data Loading & Exploration

Load the training and test datasets and perform exploratory data analysis.

In [154]:
# ── Load datasets ─────────────────────────────────────────────────
train_df = pd.read_csv('Dataset/train.csv')
test_df  = pd.read_csv('Dataset/test.csv')

print('=' * 60)
print('  DATASET OVERVIEW')
print('=' * 60)
print(f'  Training set  : {train_df.shape[0]:,} rows  ×  {train_df.shape[1]} columns')
print(f'  Test set      : {test_df.shape[0]:,} rows  ×  {test_df.shape[1]} columns')
print()

print('── Column Data Types ────────────────────────────────────')
print(train_df.dtypes.to_string())
print()

print('── Missing Values (Train) ───────────────────────────────')
missing = train_df.isnull().sum()
print(missing[missing > 0] if missing.any() else '  No missing values! ✅')
print()

print('── Sample Rows ──────────────────────────────────────────')
display(train_df.head(5))


  DATASET OVERVIEW
  Training set  : 11,128 rows  ×  15 columns
  Test set      : 2,782 rows  ×  15 columns

── Column Data Types ────────────────────────────────────
location              object
city                  object
latitude             float64
longitude            float64
price                  int64
numBathrooms           int64
numBalconies           int64
isNegotiable           int64
SecurityDeposit        int64
Status                object
Size_ft²               int64
BHK                    int64
rooms_num              int64
property_type         object
verification_days    float64

── Missing Values (Train) ───────────────────────────────
  No missing values! ✅

── Sample Rows ──────────────────────────────────────────


,location,city,latitude,longitude,price,numBathrooms,numBalconies,isNegotiable,SecurityDeposit,Status,Size_ft²,BHK,rooms_num,property_type,verification_days
0,Thane West,Mumbai,19.216236,72.980614,27000,2,0,0,0,Unfurnished,530,1,2,Apartment,10.0
1,Thane West,Mumbai,19.271788,72.967789,13500,2,0,0,0,Unfurnished,650,1,1,Apartment,1460.0
2,Airoli,Mumbai,19.143419,72.997307,20000,1,0,0,0,Semi-Furnished,700,1,1,Apartment,730.0
3,Mahipalpur,Delhi,28.547979,77.130135,14000,1,0,0,0,Furnished,1200,0,1,Studio Apartment,150.0
4,Jasola,Delhi,28.529716,77.293137,24500,2,1,0,49000,Unfurnished,1000,1,2,Apartment,180.0


In [155]:
# ── Statistical Summary ───────────────────────────────────────────
print('── Statistical Summary (numerical features) ─────────────')
display(train_df.describe().T.round(2))

print()
print('── Target Variable (price) Distribution ─────────────────')
print(f'  Min    : ₹{train_df["price"].min():>12,.0f}')
print(f'  Max    : ₹{train_df["price"].max():>12,.0f}')
print(f'  Mean   : ₹{train_df["price"].mean():>12,.0f}')
print(f'  Median : ₹{train_df["price"].median():>12,.0f}')
print(f'  Std    : ₹{train_df["price"].std():>12,.0f}')


── Statistical Summary (numerical features) ─────────────


,count,mean,std,min,25%,50%,75%,max
latitude,11128.0,22.38,4.66,18.15,18.61,19.15,28.55,30.89
longitude,11128.0,74.72,2.12,18.50,72.97,73.90,77.15,91.79
price,11128.0,107122.10,192093.93,1500.00,17500.00,33000.00,90000.00,3010101.00
numBathrooms,11128.0,2.31,1.01,0.00,2.00,2.00,3.00,10.00
numBalconies,11128.0,0.70,0.99,0.00,0.00,0.00,2.00,8.00
isNegotiable,11128.0,0.09,0.29,0.00,0.00,0.00,0.00,1.00
SecurityDeposit,11128.0,261801.44,671070.03,0.00,0.00,0.00,90000.00,11401010.00
Size_ft²,11128.0,1708.67,1659.66,150.00,710.00,1050.00,1800.00,14521.00
BHK,11128.0,0.97,0.17,0.00,1.00,1.00,1.00,1.00
rooms_num,11128.0,2.34,1.10,1.00,2.00,2.00,3.00,12.00



── Target Variable (price) Distribution ─────────────────
  Min    : ₹       1,500
  Max    : ₹   3,010,101
  Mean   : ₹     107,122
  Median : ₹      33,000
  Std    : ₹     192,094


In [156]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Combine
train_df['dataset'] = 'Train'
test_df['dataset'] = 'Test'
combined_df = pd.concat([train_df[['price', 'dataset']], 
                         test_df[['price', 'dataset']]])

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. KDE Plot
sns.kdeplot(data=combined_df, x='price', hue='dataset',
            fill=True, alpha=0.4, palette=['#3498db', '#e74c3c'], ax=axes[0, 0])
axes[0, 0].set_title('Density Comparison', fontsize=14, fontweight='bold')

# 2. Histogram
sns.histplot(data=combined_df, x='price', hue='dataset',
             bins=50, alpha=0.5, palette=['#3498db', '#e74c3c'], ax=axes[0, 1])
axes[0, 1].set_title('Histogram Comparison', fontsize=14, fontweight='bold')

# 3. Box Plot
sns.boxplot(data=combined_df, x='dataset', y='price',
            palette=['#3498db', '#e74c3c'], width=0.4, ax=axes[1, 0])
axes[1, 0].set_title('Box Plot Comparison', fontsize=14, fontweight='bold')

# 4. Violin Plot
sns.violinplot(data=combined_df, x='dataset', y='price',
               palette=['#3498db', '#e74c3c'], inner='quartile', ax=axes[1, 1])
axes[1, 1].set_title('Violin Plot Comparison', fontsize=14, fontweight='bold')

# Add statistics as text
stats_text = (
    f"Train → Mean: {train_df['price'].mean():.2f} | Median: {train_df['price'].median():.2f} | Std: {train_df['price'].std():.2f}\n"
    f"Test  → Mean: {test_df['price'].mean():.2f} | Median: {test_df['price'].median():.2f} | Std: {test_df['price'].std():.2f}"
)
fig.text(0.5, -0.02, stats_text, ha='center', fontsize=13, 
         fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('📊 Price Distribution: Train vs Test', 
             fontsize=20, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Clean up
train_df.drop('dataset', axis=1, inplace=True)
test_df.drop('dataset', axis=1, inplace=True)

In [157]:
# -- EDA Dashboard (improved) -------------------------------------------------
import numpy as np

size_col = next((c for c in train_df.columns if c.startswith('Size_ft')), 'Size_ft²')
price_low, price_high = train_df['price'].quantile([0.01, 0.99])

train_vis = train_df.copy()
test_vis = test_df.copy()
train_vis['price_clip'] = train_vis['price'].clip(price_low, price_high)
test_vis['price_clip'] = test_vis['price'].clip(price_low, price_high)

combined = pd.concat([
    train_vis[['price_clip']].rename(columns={'price_clip': 'price'}).assign(split='Train'),
    test_vis[['price_clip']].rename(columns={'price_clip': 'price'}).assign(split='Test')
], ignore_index=True)

split_palette = {'Train': '#0E7490', 'Test': '#DC6B2F'}
city_order = train_vis.groupby('city')['price'].median().sort_values(ascending=False).index.tolist()
city_colors = ['#1D4ED8', '#0F766E', '#D97706', '#BE185D', '#7C3AED', '#B45309']
city_palette = {c: city_colors[i % len(city_colors)] for i, c in enumerate(city_order)}

fig, axes = plt.subplots(2, 3, figsize=(20, 11), constrained_layout=True)
fig.patch.set_facecolor('#F8FAFC')

for ax in axes.ravel():
    ax.set_facecolor('white')
    ax.grid(alpha=0.2, linestyle='--', linewidth=0.8)

# 1) Split-wise distribution (clipped)
ax = axes[0, 0]
sns.histplot(data=combined, x='price', hue='split', bins=60, stat='density', kde=True,
             common_norm=False, alpha=0.28, palette=split_palette, edgecolor=None, ax=ax)
ax.set_title('Rent Distribution (1st-99th percentile clipped)', fontweight='bold')
ax.set_xlabel('Monthly Rent (₹)')
ax.set_ylabel('Density')

# 2) Boxen plot for spread
ax = axes[0, 1]
sns.boxenplot(data=combined, x='split', y='price', hue='split', palette=split_palette, dodge=False, ax=ax)
if ax.legend_ is not None:
    ax.legend_.remove()
ax.set_title('Rent Spread by Split', fontweight='bold')
ax.set_xlabel('Dataset Split')
ax.set_ylabel('Monthly Rent (₹)')

# 3) Median rent by city
ax = axes[0, 2]
city_med = train_vis.groupby('city', as_index=False)['price'].median().sort_values('price', ascending=False)
sns.barplot(data=city_med, x='city', y='price', hue='city', order=city_order, palette=city_palette, dodge=False, ax=ax)
if ax.legend_ is not None:
    ax.legend_.remove()
ax.set_title('Median Rent by City', fontweight='bold')
ax.set_xlabel('City')
ax.set_ylabel('Median Rent (₹)')
for t in ax.get_xticklabels():
    t.set_rotation(20)

# 4) BHK vs median rent
ax = axes[1, 0]
bhk_med = train_vis.groupby('BHK', as_index=False)['price'].median().sort_values('BHK')
sns.lineplot(data=bhk_med, x='BHK', y='price', marker='o', color='#0F766E', linewidth=2.5, ax=ax)
ax.set_title('Median Rent vs BHK', fontweight='bold')
ax.set_xlabel('BHK')
ax.set_ylabel('Median Rent (₹)')

# 5) Size vs rent (sampled)
ax = axes[1, 1]
sample_n = min(len(train_vis), 3500)
scatter = train_vis.sample(sample_n, random_state=42).copy()
scatter[size_col] = scatter[size_col].clip(upper=scatter[size_col].quantile(0.99))
scatter['price_clip'] = scatter['price'].clip(price_low, price_high)
sns.scatterplot(data=scatter, x=size_col, y='price_clip', hue='city', hue_order=city_order,
                palette=city_palette, s=22, alpha=0.55, linewidth=0, ax=ax)
ax.set_title('Size vs Rent (sampled)', fontweight='bold')
ax.set_xlabel('Size (sq ft)')
ax.set_ylabel('Monthly Rent (₹, clipped)')
ax.legend(title='City', fontsize=8, title_fontsize=9, frameon=True, loc='best')

# 6) Numeric correlation heatmap
ax = axes[1, 2]
num_cols = ['price', size_col, 'BHK', 'numBathrooms', 'numBalconies', 'rooms_num', 'SecurityDeposit', 'verification_days']
num_cols = [c for c in num_cols if c in train_vis.columns]
corr = train_vis[num_cols].corr(numeric_only=True)
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            annot_kws={'size': 8}, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Numeric Feature Correlation', fontweight='bold')

train_mean, train_med, train_std = train_df['price'].mean(), train_df['price'].median(), train_df['price'].std()
test_mean, test_med, test_std = test_df['price'].mean(), test_df['price'].median(), test_df['price'].std()
summary = (
    f"Train: mean ₹{train_mean:,.0f} | median ₹{train_med:,.0f} | std ₹{train_std:,.0f}    "
    f"Test: mean ₹{test_mean:,.0f} | median ₹{test_med:,.0f} | std ₹{test_std:,.0f}"
)
fig.suptitle('UrbanNest EDA Overview', fontsize=18, fontweight='bold', color='#0F172A')
fig.text(0.5, 0.01, summary, ha='center', fontsize=11, color='#1E293B')

plt.savefig('plots/eda_overview.png', dpi=220, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('[INFO] EDA dashboard saved: plots/eda_overview.png')


[INFO] EDA dashboard saved: plots/eda_overview.png


---
## 🔧 Section 3: Preprocessing Pipeline

**Key design decisions:**
- Use `sklearn.preprocessing.LabelEncoder` for ALL categorical features
- Save each encoder as a separate pickle file AND in a combined dictionary
- The combined `label_encoders.pkl` file is loaded by the Streamlit app

**Categorical columns:** `city`, `location`, `Status`, `property_type`

In [158]:
# ── Identify feature columns ──────────────────────────────────────
TARGET = 'price'
CATEGORICAL_COLS = ['city', 'location', 'Status', 'property_type']
NUMERICAL_COLS = [
    'latitude', 'longitude', 'numBathrooms', 'numBalconies',
    'isNegotiable', 'SecurityDeposit', 'Size_ft²', 'BHK',
    'rooms_num', 'verification_days'
]

FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS

print(f'Target variable   : {TARGET}')
print(f'Categorical cols  : {CATEGORICAL_COLS}')
print(f'Numerical cols    : {NUMERICAL_COLS}')
print(f'Total features    : {len(FEATURE_COLS)}')


Target variable   : price
Categorical cols  : ['city', 'location', 'Status', 'property_type']
Numerical cols    : ['latitude', 'longitude', 'numBathrooms', 'numBalconies', 'isNegotiable', 'SecurityDeposit', 'Size_ft²', 'BHK', 'rooms_num', 'verification_days']
Total features    : 14


In [159]:
# ── Apply LabelEncoder to each categorical column ─────────────────
# We fit on the UNION of train + test values to avoid unseen labels.

label_encoders = {}   # combined dictionary {col_name: LabelEncoder}

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    # Combine train and test to cover all known categories
    all_values = pd.concat([
        train_df[col].astype(str),
        test_df[col].astype(str)
    ]).unique()
    le.fit(all_values)

    # Transform train and test
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col]  = le.transform(test_df[col].astype(str))

    # Store encoder
    label_encoders[col] = le

    # Save individual encoder file
    individual_path = f'models/le_{col}.pkl'
    with open(individual_path, 'wb') as f:
        pickle.dump(le, f)

    print(f'  ✅  LabelEncoder for "{col}" — '
          f'{len(le.classes_)} unique classes — saved to {individual_path}')

# Save combined encoders dictionary (used by Streamlit)
with open('models/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

print()
print('✅ All label encoders saved to models/label_encoders.pkl')


  ✅  LabelEncoder for "city" — 4 unique classes — saved to models/le_city.pkl
  ✅  LabelEncoder for "location" — 702 unique classes — saved to models/le_location.pkl
  ✅  LabelEncoder for "Status" — 3 unique classes — saved to models/le_Status.pkl
  ✅  LabelEncoder for "property_type" — 6 unique classes — saved to models/le_property_type.pkl

✅ All label encoders saved to models/label_encoders.pkl


In [160]:
# ── Handle missing values (fill with median for numerical) ─────────
for col in NUMERICAL_COLS:
    if train_df[col].isnull().any():
        median_val = train_df[col].median()
        train_df[col].fillna(median_val, inplace=True)
        test_df[col].fillna(median_val, inplace=True)
        print(f'  Filled NaN in "{col}" with median = {median_val:.2f}')

# ── Feature / Target split ────────────────────────────────────────
X_train = train_df[FEATURE_COLS].copy()
y_train = train_df[TARGET].copy()

X_test  = test_df[FEATURE_COLS].copy()
y_test  = test_df[TARGET].copy()

print(f'X_train shape : {X_train.shape}')
print(f'y_train shape : {y_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'y_test  shape : {y_test.shape}')
print()
print('✅ Preprocessing complete. Data is ready for modeling.')


X_train shape : (11128, 14)
y_train shape : (11128,)
X_test  shape : (2782, 14)
y_test  shape : (2782,)

✅ Preprocessing complete. Data is ready for modeling.


---
## 📡 Section 4: Experiment Tracking Setup

We use `trackio` to track all hyperparameter optimization experiments.
All search methods are logged in a single timestamped run under `UrbanNest_Rent_Prediction`, with method-specific metric keys and trial-step curves for comparison.

In [ ]:
from datetime import datetime

project_name = "UrbanNest_Rent_Prediction"
run_name = f"rf_tuning_cv5_trials60_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
ENABLE_TRACKIO = True  # Set True only if trackio works reliably in your environment

tracker = None
if ENABLE_TRACKIO:
    try:
        if hasattr(trackio, "Experiment"):
            tracker = trackio.Experiment(project_name)
        elif hasattr(trackio, "init"):
            tracker = trackio.init(project=project_name, name=run_name, embed=False)
        else:
            print('[WARN] Unsupported trackio API. Continuing without tracking.')
            tracker = None
    except Exception as e:
        print(f'[WARN] trackio init failed: {e}. Continuing without tracking.')
        tracker = None

if tracker is not None:
    print(f'[INFO] Tracker initialized: project={project_name}, run={run_name}')
else:
    print('[WARN] Tracker disabled')

# -- Storage for results comparison --------------------------------
results_summary = {}   # {method_name: {mae, time, params}}
cv_curves       = {}   # {method_name: [best_mae_at_trial_k, ...]}

# -- 5-Fold CV configuration ---------------------------------------
N_FOLDS  = 5
N_TRIALS = 60  # same for all three methods
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f'   CV folds : {N_FOLDS}')
print(f'   Trials   : {N_TRIALS} (per method)')

# -- Task 4 logging registry (for checklist validation) ------------
trackio_log_counts = {}
trackio_log_steps = {}


def _method_slug(name: str) -> str:
    return (
        name.lower()
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('-', '_')
    )


# -- Safe logging helper (trackio API-compatible) ------------------
def log_to_trackio(metrics, params=None, step=None, prefix=None):
    if tracker is None:
        return

    payload = {}
    for k, v in dict(metrics).items():
        key = f'{prefix}_{k}' if prefix else k
        payload[key] = float(v) if isinstance(v, (int, float, np.integer, np.floating)) else v

    if params:
        for k, v in params.items():
            key = f'{prefix}_param_{k}' if prefix else f'param_{k}'
            payload[key] = float(v) if isinstance(v, (int, float, np.integer, np.floating)) else v

    try:
        if step is None:
            tracker.log(payload)
        else:
            tracker.log(payload, step=int(step))

        # Record successfully logged metric keys for Task 4 checklist
        for metric_key in payload.keys():
            trackio_log_counts[metric_key] = trackio_log_counts.get(metric_key, 0) + 1
            if step is not None:
                trackio_log_steps.setdefault(metric_key, set()).add(int(step))
    except Exception as e:
        print(f'[WARN] trackio logging skipped: {e}')


def build_best_so_far_curve(trial_mae_values):
    values = [float(v) for v in trial_mae_values]
    best_curve = []
    best = float('inf')
    for v in values:
        if v < best:
            best = v
        best_curve.append(best)
    return values, best_curve




* Created new run: rf_tuning_cv5_trials60_20260414_015441
[INFO] Tracker initialized: project=UrbanNest_Rent_Prediction, run=rf_tuning_cv5_trials60_20260414_015441
   CV folds : 5
   Trials   : 60 (per method)


---
## ⚙️ Section 5: Hyperparameter Optimization

### Search Space
| Parameter | Range |
|-----------|-------|
| `n_estimators` | 50 → 200 |
| `max_depth` | 10 → 30 |
| `min_samples_split` | 2 → 10 |

All three methods use **5-fold cross-validation** scoring with **negative MAE**.

### 5A. Grid Search (60 combinations)

In [162]:
# -- Grid Search: exactly 60 combinations --------------------------
# 4 * 5 * 3 = 60 combinations
param_grid = {
    'n_estimators':      [50, 100, 150, 200],         # 4 values
    'max_depth':         [10, 15, 20, 25, 30],        # 5 values
    'min_samples_split': [2, 5, 8],                   # 3 values
}  # Total: 4 * 5 * 3 = 60

total_combinations = (len(param_grid['n_estimators']) *
                      len(param_grid['max_depth']) *
                      len(param_grid['min_samples_split']))
print(f'[INFO] Grid Search combinations: {total_combinations} (target: 60)')
assert total_combinations == 60, 'Combination count mismatch!'

rf_base = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS)

grid_cv = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_absolute_error',
    n_jobs=N_JOBS,
    verbose=0,
    return_train_score=False
)

print('[INFO] Running Grid Search...')
t0_grid = time.time()
with parallel_backend(PARALLEL_BACKEND):
    grid_cv.fit(X_train, y_train)
elapsed_grid = time.time() - t0_grid

# Best results
grid_best_mae    = -grid_cv.best_score_
grid_best_params = grid_cv.best_params_

# Build cumulative-minimum curve for plotting
cv_results_df = pd.DataFrame(grid_cv.cv_results_)
grid_trial_maes = (-cv_results_df['mean_test_score']).tolist()
grid_trial_maes, cummin_grid = build_best_so_far_curve(grid_trial_maes)
cv_curves['Grid Search'] = cummin_grid

# Store summary
results_summary['Grid Search'] = {
    'Best MAE (CV)': grid_best_mae,
    'Time (s)': elapsed_grid,
    'Best Params': grid_best_params
}


print(f'✅ Grid Search complete!')
print(f'   Time taken   : {elapsed_grid:.1f}s')
print(f'   Best CV MAE  : ₹{grid_best_mae:,.2f}')
print(f'   Best params  : {grid_best_params}')


[INFO] Grid Search combinations: 60 (target: 60)
[INFO] Running Grid Search...
✅ Grid Search complete!
   Time taken   : 167.0s
   Best CV MAE  : ₹13,493.20
   Best params  : {'max_depth': 30, 'min_samples_split': 2, 'n_estimators': 200}


### 5B. Random Search (60 iterations)

In [163]:
# -- Random Search: 60 iterations ----------------------------------
from scipy.stats import randint

param_distributions = {
    'n_estimators':      randint(50, 201),   # 50-200 inclusive
    'max_depth':         randint(10, 31),    # 10-30 inclusive
    'min_samples_split': randint(2, 11),     # 2-10 inclusive
}

rand_cv = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
    param_distributions=param_distributions,
    n_iter=N_TRIALS,
    cv=kf,
    scoring='neg_mean_absolute_error',
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=0,
    return_train_score=False
)

print(f'[INFO] Running Random Search (n_iter={N_TRIALS})...')
t0_rand = time.time()
with parallel_backend(PARALLEL_BACKEND):
    rand_cv.fit(X_train, y_train)
elapsed_rand = time.time() - t0_rand

rand_best_mae    = -rand_cv.best_score_
rand_best_params = rand_cv.best_params_

# Build cumulative-minimum curve
rand_results_df = pd.DataFrame(rand_cv.cv_results_)
rand_trial_maes = (-rand_results_df['mean_test_score']).tolist()
rand_trial_maes, cummin_rand = build_best_so_far_curve(rand_trial_maes)
cv_curves['Random Search'] = cummin_rand

results_summary['Random Search'] = {
    'Best MAE (CV)': rand_best_mae,
    'Time (s)': elapsed_rand,
    'Best Params': rand_best_params
}


print(f'✅ Random Search complete!')
print(f'   Time taken   : {elapsed_rand:.1f}s')
print(f'   Best CV MAE  : ₹{rand_best_mae:,.2f}')
print(f'   Best params  : {rand_best_params}')


[INFO] Running Random Search (n_iter=60)...
✅ Random Search complete!
   Time taken   : 176.6s
   Best CV MAE  : ₹13,550.13
   Best params  : {'max_depth': 22, 'min_samples_split': 2, 'n_estimators': 138}


### 5C. Bayesian Optimization with Optuna (60 trials)

In [164]:
# -- Bayesian Optimization (Optuna): 60 trials ---------------------

# Track per-trial MAE for convergence plot
optuna_trial_maes = []

def objective(trial):
    """Optuna objective function: 5-fold CV MAE for RFRegressor."""
    params = {
        'n_estimators':      trial.suggest_int('n_estimators',      50,  200),
        'max_depth':         trial.suggest_int('max_depth',         10,  30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2,   10),
        'random_state':      RANDOM_STATE,
        'n_jobs':            N_JOBS,
    }
    rf = RandomForestRegressor(**params)

    # 5-fold CV: return mean MAE
    with parallel_backend(PARALLEL_BACKEND):
        cv_scores = cross_val_score(
            rf, X_train, y_train,
            cv=kf,
            scoring='neg_mean_absolute_error',
            n_jobs=N_JOBS
        )
    mae = -np.mean(cv_scores)
    optuna_trial_maes.append(mae)
    return mae


# Create Optuna study
study = optuna.create_study(
    direction='minimize',
    study_name='RentPredictor_Bayesian',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

print(f'[INFO] Running Bayesian Optimization (n_trials={N_TRIALS})...')
t0_bayes = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
elapsed_bayes = time.time() - t0_bayes

bayes_best_mae    = study.best_value
bayes_best_params = study.best_params

# Build cumulative-minimum curve
optuna_trial_maes, cummin_bayes = build_best_so_far_curve(optuna_trial_maes)
cv_curves['Bayesian (Optuna)'] = cummin_bayes

results_summary['Bayesian (Optuna)'] = {
    'Best MAE (CV)': bayes_best_mae,
    'Time (s)': elapsed_bayes,
    'Best Params': {k: v for k, v in bayes_best_params.items()}
}


print(f'✅ Bayesian Optimization complete!')
print(f'   Time taken   : {elapsed_bayes:.1f}s')
print(f'   Best CV MAE  : ₹{bayes_best_mae:,.2f}')
print(f'   Best params  : {bayes_best_params}')


[INFO] Running Bayesian Optimization (n_trials=60)...


  0%|          | 0/60 [00:00<?, ?it/s]

✅ Bayesian Optimization complete!
   Time taken   : 261.1s
   Best CV MAE  : ₹13,498.39
   Best params  : {'n_estimators': 189, 'max_depth': 30, 'min_samples_split': 2}


---
## 📈 Section 6: Convergence Analysis & Plots

Two required plots:
1. **`trials_vs_error.png`** — Convergence curve for all three methods
2. **`optuna_hyperparameter_space.png`** — Optuna exploration visualization

In [165]:
# ── Plot 1: Trials vs. Best Error (Convergence Curves) ───────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0F0F1A')
ax.set_facecolor('#1A1A2E')

COLORS_PLOT = {
    'Grid Search':        '#FF6584',
    'Random Search':      '#43CBFF',
    'Bayesian (Optuna)':  '#6C63FF',
}
MARKERS = {
    'Grid Search':        'o',
    'Random Search':      's',
    'Bayesian (Optuna)':  '^',
}

for method, curve in cv_curves.items():
    trials = list(range(1, len(curve) + 1))
    ax.plot(
        trials, curve,
        label=method,
        color=COLORS_PLOT[method],
        linewidth=2.5,
        alpha=0.9,
        marker=MARKERS[method],
        markevery=max(1, len(curve) // 10),
        markersize=6
    )

ax.set_xlabel('Trial / Iteration Number', fontsize=12, color='#A0A0C0', labelpad=8)
ax.set_ylabel('Best MAE Found So Far (₹)', fontsize=12, color='#A0A0C0', labelpad=8)
ax.set_title('Hyperparameter Optimization Convergence\n'
             'Best MAE vs. Number of Trials/Iterations',
             fontsize=14, color='white', pad=15)

ax.tick_params(colors='white', labelsize=10)
ax.spines[:].set_color('#444')
ax.grid(True, alpha=0.2, linestyle='--', color='#888')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'₹{x:,.0f}'
))

legend = ax.legend(loc='upper right', fontsize=11,
                   facecolor='#1A1A2E', edgecolor='#444',
                   labelcolor='white', framealpha=0.9)

# Annotate best values
for method, curve in cv_curves.items():
    best_trial = int(np.argmin(curve)) + 1
    best_val   = min(curve)
    ax.annotate(
        f'₹{best_val:,.0f}',
        xy=(best_trial, best_val),
        xytext=(best_trial + 2, best_val * 1.02),
        fontsize=8,
        color=COLORS_PLOT[method],
        arrowprops=dict(arrowstyle='->', color=COLORS_PLOT[method], lw=1.2)
    )

plt.tight_layout()
plt.savefig('plots/trials_vs_error.png', bbox_inches='tight', dpi=120,
            facecolor=fig.get_facecolor())
plt.show()
print('✅ Convergence plot saved: plots/trials_vs_error.png')


✅ Convergence plot saved: plots/trials_vs_error.png


In [166]:
# ── Plot 2: Optuna Hyperparameter Space Visualization ────────────
try:
    import plotly
    from optuna.visualization import plot_optimization_history, plot_contour

    # Option A: Optimization history (interactive HTML version)
    fig_history = plot_optimization_history(study)
    fig_history.update_layout(
        paper_bgcolor='#0F0F1A',
        plot_bgcolor='#1A1A2E',
        font=dict(color='white', size=12),
        title_font=dict(size=14)
    )
    fig_history.write_html('plots/optuna_history.html')

    # Save as static PNG using matplotlib fallback
    trials_df = study.trials_dataframe()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor('#0F0F1A')
    fig.suptitle('Optuna — Hyperparameter Space Exploration', fontsize=14,
                 color='white', y=1.01)

    # Subplot 1: n_estimators vs MAE (scatter)
    ax = axes[0]
    ax.set_facecolor('#1A1A2E')
    sc = ax.scatter(
        trials_df['params_n_estimators'],
        trials_df['value'],
        c=trials_df['params_max_depth'],
        cmap='plasma', alpha=0.75, s=60, edgecolors='none'
    )
    plt.colorbar(sc, ax=ax, label='max_depth')
    ax.set_xlabel('n_estimators', color='#A0A0C0', fontsize=11)
    ax.set_ylabel('Trial MAE (₹)', color='#A0A0C0', fontsize=11)
    ax.set_title('n_estimators vs MAE\n(colour = max_depth)',
                 color='white', fontsize=12)
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#444')
    ax.grid(True, alpha=0.2, linestyle='--', color='#888')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'₹{x:,.0f}'
    ))

    # Subplot 2: max_depth vs min_samples_split (heat-like scatter)
    ax = axes[1]
    ax.set_facecolor('#1A1A2E')
    sc2 = ax.scatter(
        trials_df['params_max_depth'],
        trials_df['params_min_samples_split'],
        c=trials_df['value'],
        cmap='viridis', alpha=0.75, s=60, edgecolors='none'
    )
    plt.colorbar(sc2, ax=ax, label='MAE (₹)')
    ax.set_xlabel('max_depth', color='#A0A0C0', fontsize=11)
    ax.set_ylabel('min_samples_split', color='#A0A0C0', fontsize=11)
    ax.set_title('2D Hyperparameter Space\n(max_depth vs min_samples_split)',
                 color='white', fontsize=12)
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#444')
    ax.grid(True, alpha=0.2, linestyle='--', color='#888')

    plt.tight_layout()
    plt.savefig('plots/optuna_hyperparameter_space.png', bbox_inches='tight',
                dpi=120, facecolor=fig.get_facecolor())
    plt.show()
    print('✅ Optuna hyperparameter space plot saved: plots/optuna_hyperparameter_space.png')

except ImportError:
    # Fallback without plotly
    print('⚠️  plotly not available — generating matplotlib fallback')
    trials_df = study.trials_dataframe()

    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('#0F0F1A')
    ax.set_facecolor('#1A1A2E')

    sc = ax.scatter(
        trials_df['params_n_estimators'],
        trials_df['value'],
        c=trials_df['params_max_depth'],
        cmap='plasma', alpha=0.8, s=70
    )
    plt.colorbar(sc, label='max_depth')
    ax.set_xlabel('n_estimators', color='#A0A0C0')
    ax.set_ylabel('MAE (₹)', color='#A0A0C0')
    ax.set_title('Optuna: Hyperparameter Space', color='white')
    ax.tick_params(colors='white')
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig('plots/optuna_hyperparameter_space.png', bbox_inches='tight',
                dpi=120, facecolor=fig.get_facecolor())
    plt.show()
    print('✅ Fallback plot saved: plots/optuna_hyperparameter_space.png')


✅ Optuna hyperparameter space plot saved: plots/optuna_hyperparameter_space.png


---
## 🏆 Section 7: Final Model Training & Evaluation

Select the globally best hyperparameters from all three methods, train the final model on the **complete** training set, then evaluate once on the **held-out test set**.

In [167]:
# ── Select best hyperparameters globally ─────────────────────────
all_methods = {
    'Grid Search':       (grid_best_mae,  grid_best_params),
    'Random Search':     (rand_best_mae,  rand_best_params),
    'Bayesian (Optuna)': (bayes_best_mae, bayes_best_params),
}

best_method = min(all_methods, key=lambda k: all_methods[k][0])
best_cv_mae, best_params_raw = all_methods[best_method]

# Extract only the RF parameters (exclude 'random_state' if Bayesian added it)
best_final_params = {
    'n_estimators':      best_params_raw['n_estimators'],
    'max_depth':         best_params_raw['max_depth'],
    'min_samples_split': best_params_raw['min_samples_split'],
    'random_state':      RANDOM_STATE,
    'n_jobs':            N_JOBS,
}

print(f'🏆 Best method: {best_method}')
print(f'   CV MAE     : ₹{best_cv_mae:,.2f}')
print(f'   Params     : {best_final_params}')


🏆 Best method: Grid Search
   CV MAE     : ₹13,493.20
   Params     : {'n_estimators': 200, 'max_depth': 30, 'min_samples_split': 2, 'random_state': 42, 'n_jobs': -1}


In [168]:
# -- Train final model on full training set ------------------------
print('[INFO] Training final model on full training data...')
t0_final = time.time()

final_model = RandomForestRegressor(**best_final_params)
final_model.fit(X_train, y_train)

elapsed_final = time.time() - t0_final
print(f'[INFO] Final model trained in {elapsed_final:.1f}s')

# -- Evaluate on TEST set (only done once) -------------------------
y_pred_test = final_model.predict(X_test)
test_mae    = mean_absolute_error(y_test, y_pred_test)

print(f'[INFO] FINAL TEST MAE: ₹{test_mae:,.2f}')

# -- Save final model ----------------------------------------------
model_path = 'models/best_rf_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(final_model, f)

# Also save with joblib for efficiency
joblib.dump(final_model, 'models/best_rf_model.joblib')

print(f'✅ Final model saved to: {model_path}')

print(f'[INFO] Final model saved to: {model_path}')



# Update results summary
results_summary['Final Test'] = {
    'MAE (Test Set)': test_mae,
    'Best Method':    best_method,
    'Model':         'RandomForestRegressor',
}


[INFO] Training final model on full training data...
[INFO] Final model trained in 1.3s
[INFO] FINAL TEST MAE: ₹12,414.25
✅ Final model saved to: models/best_rf_model.pkl
[INFO] Final model saved to: models/best_rf_model.pkl


### 7A. Trackio Logging (Single Cell)

All Trackio metric logging is centralized here for readability and easier debugging.


In [169]:
# -- Centralized Trackio Logging + Task 4 Checklist -------------------
# This is the ONLY cell that writes metrics to Trackio.
from sklearn.metrics import r2_score
from optuna.importance import get_param_importances


def _safe_key(text: str) -> str:
    return (
        str(text)
        .strip()
        .lower()
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('-', '_')
        .replace('?', '2')
    )


if tracker is not None:
    # Reset registry so reruns of this cell do not double-count checklist stats.
    trackio_log_counts = {}
    trackio_log_steps = {}

    # -----------------------------------------------------------------
    # (1) Model Comparison (MAE / R?) + (2) Time vs Performance tradeoff
    # -----------------------------------------------------------------
    model_cfgs = {
        'grid': {
            'params': {
                'n_estimators': int(grid_best_params['n_estimators']),
                'max_depth': int(grid_best_params['max_depth']),
                'min_samples_split': int(grid_best_params['min_samples_split']),
                'random_state': RANDOM_STATE,
                'n_jobs': N_JOBS,
            },
            'train_time_s': float(elapsed_grid),
            'best_cv_mae': float(grid_best_mae),
            'trials': int(total_combinations),
        },
        'random': {
            'params': {
                'n_estimators': int(rand_best_params['n_estimators']),
                'max_depth': int(rand_best_params['max_depth']),
                'min_samples_split': int(rand_best_params['min_samples_split']),
                'random_state': RANDOM_STATE,
                'n_jobs': N_JOBS,
            },
            'train_time_s': float(elapsed_rand),
            'best_cv_mae': float(rand_best_mae),
            'trials': int(N_TRIALS),
        },
        'bayes': {
            'params': {
                'n_estimators': int(bayes_best_params['n_estimators']),
                'max_depth': int(bayes_best_params['max_depth']),
                'min_samples_split': int(bayes_best_params['min_samples_split']),
                'random_state': RANDOM_STATE,
                'n_jobs': N_JOBS,
            },
            'train_time_s': float(elapsed_bayes),
            'best_cv_mae': float(bayes_best_mae),
            'trials': int(N_TRIALS),
        },
    }

    method_test_metrics = {}

    for method_key, cfg in model_cfgs.items():
        mdl = RandomForestRegressor(**cfg['params'])
        mdl.fit(X_train, y_train)
        y_pred = mdl.predict(X_test)

        method_test_mae = float(mean_absolute_error(y_test, y_pred))
        method_test_r2 = float(r2_score(y_test, y_pred))

        method_test_metrics[method_key] = {
            'model': mdl,
            'y_pred': y_pred,
            'test_mae': method_test_mae,
            'test_r2': method_test_r2,
        }

        # Method-level scalar metrics
        log_to_trackio(
            metrics={
                f'search_best_cv_mae_{method_key}': cfg['best_cv_mae'],
                f'search_time_seconds_{method_key}': cfg['train_time_s'],
                f'search_trials_count_{method_key}': cfg['trials'],
                f'test_mae_{method_key}': method_test_mae,
                f'test_r2_{method_key}': method_test_r2,
            },
            params=cfg['params'],
        )

    # Time vs performance scatter series (step map: 1=grid, 2=random, 3=bayes)
    step_map = {'grid': 1, 'random': 2, 'bayes': 3}
    for method_key, step_no in step_map.items():
        log_to_trackio(
            metrics={
                'tradeoff_time_seconds': float(model_cfgs[method_key]['train_time_s']),
                'tradeoff_test_mae': float(method_test_metrics[method_key]['test_mae']),
                'tradeoff_test_r2': float(method_test_metrics[method_key]['test_r2']),
            },
            step=step_no,
        )

    # -----------------------------------------------------------------
    # (3) Hyperparameter Optimization History (Optuna)
    # -----------------------------------------------------------------
    combined_trials = min(
        len(grid_trial_maes), len(rand_trial_maes), len(optuna_trial_maes),
        len(cummin_grid), len(cummin_rand), len(cummin_bayes)
    )

    for step_idx in range(combined_trials):
        step_no = step_idx + 1
        log_to_trackio(
            {
                'cmp_best_mae_grid': float(cummin_grid[step_idx]),
                'cmp_best_mae_random': float(cummin_rand[step_idx]),
                'cmp_best_mae_bayes': float(cummin_bayes[step_idx]),
                'cmp_trial_mae_grid': float(grid_trial_maes[step_idx]),
                'cmp_trial_mae_random': float(rand_trial_maes[step_idx]),
                'cmp_trial_mae_bayes': float(optuna_trial_maes[step_idx]),
                'optuna_history_best_mae': float(cummin_bayes[step_idx]),
                'optuna_history_trial_mae': float(optuna_trial_maes[step_idx]),
            },
            step=step_no,
        )

    # -----------------------------------------------------------------
    # (4) Parameter Importance (Optuna)
    # -----------------------------------------------------------------
    try:
        param_importance = get_param_importances(study)
    except Exception as e:
        print(f'[WARN] Optuna parameter importance skipped: {e}')
        param_importance = {}

    if param_importance:
        for rank, (param_name, imp) in enumerate(param_importance.items(), start=1):
            safe_name = _safe_key(param_name)
            log_to_trackio(
                metrics={
                    'optuna_param_importance': float(imp),
                    f'optuna_param_importance_{safe_name}': float(imp),
                },
                step=rank,
            )

    # -----------------------------------------------------------------
    # (5) Actual vs Predicted + (6) Residual Plot (final model)
    # -----------------------------------------------------------------
    actual_vals = np.asarray(y_test)
    pred_vals = np.asarray(y_pred_test)
    residual_vals = actual_vals - pred_vals

    max_points = min(300, len(actual_vals))
    sample_idx = np.linspace(0, len(actual_vals) - 1, num=max_points, dtype=int)

    for step_no, idx in enumerate(sample_idx, start=1):
        log_to_trackio(
            metrics={
                'actual_vs_pred_actual': float(actual_vals[idx]),
                'actual_vs_pred_predicted': float(pred_vals[idx]),
                'residual_plot_predicted': float(pred_vals[idx]),
                'residual_plot_residual': float(residual_vals[idx]),
            },
            step=step_no,
        )

    # -----------------------------------------------------------------
    # (7) Feature Importance (Random Forest final model)
    # -----------------------------------------------------------------
    if hasattr(final_model, 'feature_importances_') and hasattr(final_model, 'feature_names_in_'):
        importances = list(zip(final_model.feature_names_in_, final_model.feature_importances_))
        importances.sort(key=lambda x: x[1], reverse=True)

        top_k = min(15, len(importances))
        for rank, (feat_name, imp) in enumerate(importances[:top_k], start=1):
            safe_feat = _safe_key(feat_name)
            log_to_trackio(
                metrics={
                    'rf_feature_importance': float(imp),
                    f'rf_feature_importance_{safe_feat}': float(imp),
                },
                step=rank,
            )

    # -----------------------------------------------------------------
    # (8) Cross-validation Performance (fold-wise MAE/R?)
    # -----------------------------------------------------------------
    for method_key, cfg in model_cfgs.items():
        fold_maes = []
        fold_r2s = []

        for tr_idx, va_idx in kf.split(X_train):
            X_tr = X_train.iloc[tr_idx] if hasattr(X_train, 'iloc') else X_train[tr_idx]
            X_va = X_train.iloc[va_idx] if hasattr(X_train, 'iloc') else X_train[va_idx]
            y_tr = y_train.iloc[tr_idx] if hasattr(y_train, 'iloc') else y_train[tr_idx]
            y_va = y_train.iloc[va_idx] if hasattr(y_train, 'iloc') else y_train[va_idx]

            mdl = RandomForestRegressor(**cfg['params'])
            mdl.fit(X_tr, y_tr)
            y_va_pred = mdl.predict(X_va)

            fold_maes.append(float(mean_absolute_error(y_va, y_va_pred)))
            fold_r2s.append(float(r2_score(y_va, y_va_pred)))

        for fold_no, (mae_v, r2_v) in enumerate(zip(fold_maes, fold_r2s), start=1):
            log_to_trackio(
                metrics={
                    f'cv_fold_mae_{method_key}': mae_v,
                    f'cv_fold_r2_{method_key}': r2_v,
                },
                step=fold_no,
            )

        log_to_trackio(
            metrics={
                f'cv_mean_mae_{method_key}': float(np.mean(fold_maes)),
                f'cv_std_mae_{method_key}': float(np.std(fold_maes)),
                f'cv_mean_r2_{method_key}': float(np.mean(fold_r2s)),
                f'cv_std_r2_{method_key}': float(np.std(fold_r2s)),
            }
        )

    # Final model summary + method selection consistency
    log_to_trackio(
        metrics={
            'final_model_test_mae': float(mean_absolute_error(y_test, y_pred_test)),
            'selected_method_best_cv_mae': float(best_cv_mae),
            'cmp_runtime_seconds_grid': float(elapsed_grid),
            'cmp_runtime_seconds_random': float(elapsed_rand),
            'cmp_runtime_seconds_bayes': float(elapsed_bayes),
            'cmp_best_cv_mae_grid': float(grid_best_mae),
            'cmp_best_cv_mae_random': float(rand_best_mae),
            'cmp_best_cv_mae_bayes': float(bayes_best_mae),
        },
        params=best_final_params,
    )

    method_mae_map = {
        'grid_search': grid_best_mae,
        'random_search': rand_best_mae,
        'bayesian_optuna': bayes_best_mae,
    }
    best_method_key = min(method_mae_map, key=method_mae_map.get)
    log_to_trackio({'selected_method_matches_best': 1.0 if best_method_key == _method_slug(best_method) else 0.0})

    if hasattr(tracker, 'finish'):
        tracker.finish()
        print('[INFO] Trackio run finished and persisted.')

# -- Task 4 checklist ------------------------------------------------
required_metrics = [
    'search_best_cv_mae_grid',
    'search_best_cv_mae_random',
    'search_best_cv_mae_bayes',
    'test_mae_grid',
    'test_mae_random',
    'test_mae_bayes',
    'test_r2_grid',
    'test_r2_random',
    'test_r2_bayes',
    'tradeoff_time_seconds',
    'tradeoff_test_mae',
    'optuna_history_best_mae',
    'optuna_param_importance',
    'actual_vs_pred_actual',
    'actual_vs_pred_predicted',
    'residual_plot_predicted',
    'residual_plot_residual',
    'rf_feature_importance',
    'cv_fold_mae_grid',
    'cv_fold_mae_random',
    'cv_fold_mae_bayes',
    'final_model_test_mae',
    'selected_method_matches_best',
]

print('=' * 88)
print('TASK 4 CHECKLIST - TRACKIO METRIC COVERAGE')
print('=' * 88)
print(f'Project : {project_name}')
print(f'Run     : {run_name}')
print(f'Logging : {"ENABLED" if tracker is not None else "DISABLED"}')
print()

missing_required = [m for m in required_metrics if m not in trackio_log_counts]

print('[A] Required metrics')
for m in required_metrics:
    status = 'OK' if m in trackio_log_counts else 'MISSING'
    count = trackio_log_counts.get(m, 0)
    print(f'  - {m:<45} {status:<8} count={count}')

print()
print('[B] Core series step checks')
series_checks = {
    'cmp_best_mae_bayes': N_TRIALS,
    'cmp_trial_mae_bayes': N_TRIALS,
    'optuna_history_best_mae': N_TRIALS,
    'tradeoff_time_seconds': 3,
    'cv_fold_mae_grid': N_FOLDS,
    'cv_fold_mae_random': N_FOLDS,
    'cv_fold_mae_bayes': N_FOLDS,
}
for metric_name, expected_max_step in series_checks.items():
    steps = sorted(trackio_log_steps.get(metric_name, set()))
    max_step = steps[-1] if steps else 0
    status = 'OK' if max_step == expected_max_step else 'CHECK'
    print(f'  - {metric_name:<45} {status:<8} max_step={max_step} expected={expected_max_step}')

print()
print('Summary:')
if tracker is None:
    print('  - Trackio tracker is disabled, so no external logs are expected.')
elif not missing_required:
    print('  - All required Task 4 metrics are present.')
else:
    print(f'  - Missing required metrics: {missing_required}')


* Run finished. Uploading logs to Trackio (please wait...)
[INFO] Trackio run finished and persisted.
TASK 4 CHECKLIST - TRACKIO METRIC COVERAGE
Project : UrbanNest_Rent_Prediction
Run     : rf_tuning_cv5_trials60_20260414_015441
Logging : ENABLED

[A] Required metrics
  - search_best_cv_mae_grid                       OK       count=1
  - search_best_cv_mae_random                     OK       count=1
  - search_best_cv_mae_bayes                      OK       count=1
  - test_mae_grid                                 OK       count=1
  - test_mae_random                               OK       count=1
  - test_mae_bayes                                OK       count=1
  - test_r2_grid                                  OK       count=1
  - test_r2_random                                OK       count=1
  - test_r2_bayes                                 OK       count=1
  - tradeoff_time_seconds                         OK       count=3
  - tradeoff_test_mae                             OK       c

---
## 📋 Section 8: Results Summary

In [170]:
# ── Comprehensive Results Comparison Table ────────────────────────
print('=' * 80)
print('  HYPERPARAMETER OPTIMIZATION — RESULTS SUMMARY')
print('=' * 80)
print(f'{"Method":<22} {"Best CV MAE (₹)":>18} {"Time (s)":>12} {"Best n_est":>12} {"Best depth":>12} {"Best split":>12}')
print('-' * 80)

for method in ['Grid Search', 'Random Search', 'Bayesian (Optuna)']:
    info   = results_summary[method]
    p      = info['Best Params']
    print(f'{method:<22} '
          f'{info["Best MAE (CV)"]:>18,.2f} '
          f'{info["Time (s)"]:>12.1f} '
          f'{p.get("n_estimators", "N/A"):>12} '
          f'{p.get("max_depth", "N/A"):>12} '
          f'{p.get("min_samples_split", "N/A"):>12}')

print('=' * 80)
print(f'  🏆 Winner        : {best_method}')
print(f'  🎯 Final Test MAE: ₹{test_mae:,.2f}')
print('=' * 80)


  HYPERPARAMETER OPTIMIZATION — RESULTS SUMMARY
Method                    Best CV MAE (₹)     Time (s)   Best n_est   Best depth   Best split
--------------------------------------------------------------------------------
Grid Search                     13,493.20        167.0          200           30            2
Random Search                   13,550.13        176.6          138           22            2
Bayesian (Optuna)               13,498.39        261.1          189           30            2
  🏆 Winner        : Grid Search
  🎯 Final Test MAE: ₹12,414.25


### Hyperparameter Search Strategies

| Strategy            | Key Idea                                              | When to Use                          |
|--------------------|-------------------------------------------------------|--------------------------------------|
| Grid search        | Try every combination                                 | 2–3 params, small grid               |
| Random search      | Sample randomly, better coverage                      | Quick exploration, 4+ params         |
| Bayesian (Optuna)  | Learn from past results, focus on promising areas      | Serious tuning, expensive evaluations|
| AutoML             | Loop over model families + tuning                     | Find the performance ceiling         |

In [171]:
# ── Final Verification — List saved files ─────────────────────────
print('📁 Saved model artifacts:')
for fname in sorted(os.listdir('models')):
    fpath = os.path.join('models', fname)
    print(f'   {fname:<35}  ({os.path.getsize(fpath):,} bytes)')

print()
print('📁 Saved plot files:')
for fname in sorted(os.listdir('plots')):
    fpath = os.path.join('plots', fname)
    print(f'   {fname:<35}  ({os.path.getsize(fpath):,} bytes)')

print()
print('🚀 Pipeline complete! The following files are ready for deployment:')
print('   • models/best_rf_model.pkl    → Used by app.py')
print('   • models/label_encoders.pkl   → Used by app.py')
print('   • plots/trials_vs_error.png   → Convergence analysis')
print('   • plots/optuna_hyperparameter_space.png → Bayesian search visualization')


📁 Saved model artifacts:
   best_rf_model.joblib                 (148,741,393 bytes)
   best_rf_model.pkl                    (148,731,571 bytes)
   label_encoders.pkl                   (11,429 bytes)
   le_Status.pkl                        (286 bytes)
   le_city.pkl                          (275 bytes)
   le_location.pkl                      (11,075 bytes)
   le_property_type.pkl                 (334 bytes)

📁 Saved plot files:
   eda_overview.png                     (705,999 bytes)
   optuna_history.html                  (4,856,883 bytes)
   optuna_hyperparameter_space.png      (113,256 bytes)
   trials_vs_error.png                  (87,590 bytes)

🚀 Pipeline complete! The following files are ready for deployment:
   • models/best_rf_model.pkl    → Used by app.py
   • models/label_encoders.pkl   → Used by app.py
   • plots/trials_vs_error.png   → Convergence analysis
   • plots/optuna_hyperparameter_space.png → Bayesian search visualization
